# Identifying Splice-Variants and Post-Prenylation Processing Variants in MaxQuant Results
The Protein Groups output tables from MaxQuant will be filtered by the fasta identifiers for isoforms (CTERM_VAR, INTERNAL_VAR) and prenylation variants (INTERNAL_PAE0, INTERNAL_LOWPAE und INTERNAL_C1).

# Setup

## Import and install Packages

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
import re
import openpyxl
from functions import read_fastafile

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/05_MaxQuant"):
    print("WARNING: The working directory is not set to the '05_MaxQuant'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '05_MaxQuant' folder (\"{cwd}\").")

In [ ]:
# Data directories
prenylation_dir = "../04_Prenylation_Database/data/"
mapping_dir = "../01_UniProt/data/mapping/"
isoform_database_dir = "../02_Isoform_Database/"

total_extracts_dir = "data/total_extracts/"
prenylation_enrichment_dir = "data/prenylation_enrichment/"

## Read in MaxQuant Results and Lists of isoforms and prenylated proteins

In [ ]:
mq_results_1 = pd.read_excel('data/proteinGroup_searchdb_1.xlsx') # used unique isoforms but not unique canonical

#total extracts
mq_results_act_te = pd.read_excel(os.path.join(total_extracts_dir, 'proteinGroups_act_te_nrdbln.xlsx')) # Th1-Cells activated vs non-activated
mq_results_stat_te = pd.read_excel(os.path.join(total_extracts_dir, 'proteinGroups_stat_te_nrdbln.xlsx')) # activated Th1 cells +/- statin treatment
mq_results_FTI_te = pd.read_excel(os.path.join(total_extracts_dir, 'proteinGroups_FTI_te_nrdbln.xlsx')) # activated Th1-Cells +/- treatment with farnesyl transferase inhibitor

#prenylation enriched (with clickit-reaction)
mq_results_act_clickit = pd.read_excel(os.path.join(prenylation_enrichment_dir, 'protein_Groups_act_clickit_nrdbln.xlsx')) # Th1-Cells activated vs non-activated
mq_results_stat_clickit = pd.read_excel(os.path.join(prenylation_enrichment_dir, 'protein_Groups_stat_clickit_nrdbln.xlsx')) # activated Th1 cells +/- statin treatment
mq_results_FTI_clickit = pd.read_excel(os.path.join(prenylation_enrichment_dir, 'protein_Groups_FTI_clickit_nrdbln.xlsx')) # activated Th1-Cells +/- treatment with farnesyl transferase inhibitor

isoform_fasta = read_fastafile(os.path.join(isoform_database_dir, 'TryP_Isoform_Database_Final.fasta'))
prenylation_fasta = read_fastafile(os.path.join(prenylation_dir, 'Post-Prenylation_Processing_Database.fasta'))
iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv')) 

# Filter Fasta headers by Isoform and Prenylation fasta headers

In [ ]:
isoform_set = set(isoform_fasta['seqID'])
prenylation_set = set(prenylation_fasta['seqID'])

In [ ]:
import pandas as pd

def filter_maxquant_variants(df, isoform_set, prenylation_set, 
                             id_col='Protein IDs', 
                             header_col='Fasta headers',
                             fdr_cutoff=1, 
                             min_unique_peptides=1):
    """
    Cleans MaxQuant results and splits them into Isoforms and Prenylation variants
    using statistical thresholds for FDR and peptide count.
    """
    # 1. Basic Cleaning (Decoys, Contaminants, and Only identified by site)
    initial_count = len(df)
    
    # Remove rows based on MaxQuant flag columns
    flags = ['Reverse', 'Potential contaminant', 'Only identified by site']
    for col in flags:
        if col in df.columns:
            df = df[df[col] != '+']
            
    # 2. Statistical Filtering (FDR and Unique Peptides)
    # Filter by Q-value (FDR)
    if 'Q-value' in df.columns:
        df = df[df['Q-value'] <= fdr_cutoff]
    
    # Filter by Unique Peptides
    if 'Unique peptides' in df.columns:
        df = df[df['Unique peptides'] >= min_unique_peptides]

    clean_df = df.copy()
    
    # 3. Helper function for set intersection
    def check_match(header_str, target_set):
        if pd.isna(header_str): return False
        # Split by semicolon as MaxQuant often concatenates IDs
        return any(h.strip() in target_set for h in str(header_str).split(';'))

    # 4. Create the two filtered DataFrames
    df_isoforms = clean_df[
        clean_df[header_col].apply(lambda x: check_match(x, isoform_set))
    ].copy()
    
    df_prenylation = clean_df[
        clean_df[header_col].apply(lambda x: check_match(x, prenylation_set))
    ].copy()

    # Reporting
    print(f"--- Filter Report ---")
    print(f"Initial Rows: {initial_count}")
    print(f"Passed Stats Filter (FDR <={fdr_cutoff}, Unique Peps >= {min_unique_peptides}): {len(clean_df)}")
    print(f"Isoform hits: {len(df_isoforms)}")
    print(f"Prenylation hits: {len(df_prenylation)}")
    
    return df_isoforms, df_prenylation

In [ ]:
# Execution:
isoforms_df_1, prenylation_df_1 = filter_maxquant_variants(mq_results_1, isoform_set, prenylation_set)

#total extracts
isoforms_act_te, prenylation_act_te = filter_maxquant_variants(mq_results_act_te, isoform_set, prenylation_set)
isoforms_stat_te, prenylation_stat_te = filter_maxquant_variants(mq_results_stat_te, isoform_set, prenylation_set)
isoforms_FTI_te, prenylation_FTI_te = filter_maxquant_variants(mq_results_FTI_te, isoform_set, prenylation_set)

#prentylation enrichment
isoforms_act_clickit, prenylation_act_clickit = filter_maxquant_variants(mq_results_act_clickit, isoform_set, prenylation_set)
isoforms_stat_clickit, prenylation_stat_clickit = filter_maxquant_variants(mq_results_stat_clickit, isoform_set, prenylation_set)
isoforms_FTI_clickit, prenylation_FTI_clickit = filter_maxquant_variants(mq_results_FTI_clickit, isoform_set, prenylation_set)

In [ ]:
# Save splice variants as csv 
isoforms_df_1.to_csv('data/MQ_splice_variants_1.csv', index=False)

isoforms_act_te.to_csv(os.path.join(total_extracts_dir, 'MQ_splice_variants_act_te.csv'), index=False)
isoforms_stat_te.to_csv(os.path.join(total_extracts_dir, 'MQ_splice_variants_stat_te.csv'), index=False)
isoforms_FTI_te.to_csv(os.path.join(total_extracts_dir, 'MQ_splice_variants_FTI_te.csv'), index=False)

isoforms_act_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_splice_variants_act_clickit.csv'), index=False)
isoforms_stat_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_splice_variants_stat_clickit.csv'), index=False)
isoforms_FTI_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_splice_variants_FTI_clickit.csv'), index=False)

In [ ]:
print(len(isoforms_act_te))
print(len(isoforms_stat_te))
print(len(isoforms_FTI_te))

print(len(isoforms_act_clickit))
print(len(isoforms_stat_clickit))
print(len(isoforms_FTI_clickit))

# Extract UniProt IDs 
The UniProt IDs are extracted to look at the isoforms in STRING.

Turns out STRING doesn't know isoforms so this is just kept as a function to extract IDs but not to analyze anything in STRING.

In [ ]:
def extract_uniprot_ids(df, id_col='Protein IDs'):
    """
    Extracts UniProt accessions (e.g., A0FGR8 or A0FGR8-4) 
    by targeting the second field in the ID string.
    """
    # 1. Split grouped IDs into individual entries (handle the semicolon)
    all_entries = df[id_col].str.split(';').explode()
    
    def parse_pipe_string(s):
        if pd.isna(s): return None
        parts = str(s).split('|')
        # If format is sp|A0FGR8|NAME, parts[1] is 'A0FGR8'
        # If format is just A0FGR8, parts[0] is 'A0FGR8'
        return parts[1] if len(parts) > 1 else parts[0]

    # 2. Apply the parsing logic
    extracted = all_entries.apply(parse_pipe_string)
    
    # 3. Remove duplicates and NaNs for a clean list
    unique_isoform_ids = extracted.dropna().unique().tolist()
    
    return unique_isoform_ids

In [ ]:
# Execution
IDs_isoforms_stat_te = extract_uniprot_ids(isoforms_stat_te)
IDs_isoforms_act_te = extract_uniprot_ids(isoforms_act_te)
IDs_isoforms_FTI_te = extract_uniprot_ids(isoforms_FTI_te)

IDs_isoforms_stat_clickit = extract_uniprot_ids(isoforms_stat_clickit)
IDs_isoforms_act_clickit = extract_uniprot_ids(isoforms_act_clickit)
IDs_isoforms_FTI_clickit = extract_uniprot_ids(isoforms_FTI_clickit)

# You can now print these and copy-paste them into the STRING "Multiple Proteins" search box
print(f"Total unique IDs in stat_te df: {len(IDs_isoforms_stat_te)}")
print("--------------------")
print(f"Total unique IDs in act_te df: {len(IDs_isoforms_act_te)}")
print("--------------------")
print(f"Total unique IDs in FTI_te df: {len(IDs_isoforms_FTI_te)}")
print("--------------------")

print(f"Total unique IDs in stat_te df: {len(IDs_isoforms_stat_clickit)}")
print("--------------------")
print(f"Total unique IDs in act_te df: {len(IDs_isoforms_act_clickit)}")
print("--------------------")
print(f"Total unique IDs in FTI_te df: {len(IDs_isoforms_FTI_clickit)}")

In [ ]:
# Filter out canonical forms to only keep isoform IDs that were found
canonical_ids = set(iso_canonical_mapping['Entry'])

IDs_isoforms_stat_te = [protein_id for protein_id in IDs_isoforms_stat_te if protein_id not in canonical_ids]
IDs_isoforms_act_te = [protein_id for protein_id in IDs_isoforms_act_te if protein_id not in canonical_ids]
IDs_isoforms_FTI_te = [protein_id for protein_id in IDs_isoforms_FTI_te if protein_id not in canonical_ids]

IDs_isoforms_stat_clickit = [protein_id for protein_id in IDs_isoforms_stat_clickit if protein_id not in canonical_ids]
IDs_isoforms_act_clickit = [protein_id for protein_id in IDs_isoforms_act_clickit if protein_id not in canonical_ids]
IDs_isoforms_FTI_clickit = [protein_id for protein_id in IDs_isoforms_FTI_clickit if protein_id not in canonical_ids]

# Reporting
print(f"Non-canonical Isoforms in stat_te df: {len(IDs_isoforms_stat_te)}")
print(f"Non-canonical Isoforms in act_te df: {len(IDs_isoforms_act_te)}")
print(f"Non-canonical Isoforms in FTI_te df: {len(IDs_isoforms_FTI_te)}")

print(f"Non-canonical Isoforms in stat_clickit df: {len(IDs_isoforms_stat_clickit)}")
print(f"Non-canonical Isoforms in act_clickit df: {len(IDs_isoforms_act_clickit)}")
print(f"Non-canonical Isoforms in FTI_clickit df: {len(IDs_isoforms_FTI_clickit)}")

In [ ]:
overlap_act_stat_te = list(set(IDs_isoforms_stat_te) & set(IDs_isoforms_act_te))
print(len(overlap_act_stat_te)) 

overlap_stat_FTI_te = list(set(IDs_isoforms_stat_te) & set(IDs_isoforms_FTI_te))
print(len(overlap_stat_FTI_te)) 

overlap_act_FTI_te = list(set(IDs_isoforms_act_te) & set(IDs_isoforms_FTI_te))
print(len(overlap_act_FTI_te)) 

In [ ]:
prenylation_stat_clickit